# Exploring `mujoco/invertedpendulum/expert-v0` with Minari

This notebook loads the Minari expert dataset for the MuJoCo InvertedPendulum environment,
assembles a **100 000 × 5** observation-action matrix, and lays the groundwork for training
a Self-Organizing Map (SOM) on the resulting data.

| Column index | Description |
|---|---|
| 0 | Cart position |
| 1 | Cart velocity |
| 2 | Pole angle |
| 3 | Pole angular velocity |
| 4 | Action (force applied to cart) |

## 1  Imports

In [1]:
import numpy as np
import minari
from minisom import MiniSom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## 2  Load dataset

If the dataset has not been downloaded yet, Minari will fetch it automatically from the remote
registry the first time `load_dataset` is called.

In [2]:
DATASET_ID = "mujoco/invertedpendulum/expert-v0"

# Download if not already cached locally
if DATASET_ID not in minari.list_local_datasets():
    minari.download_dataset(DATASET_ID)

dataset = minari.load_dataset(DATASET_ID)
print(f"Dataset loaded: {DATASET_ID}")
print(f"Total episodes : {dataset.total_episodes}")
print(f"Total steps    : {dataset.total_steps}")

Dataset loaded: mujoco/invertedpendulum/expert-v0
Total episodes : 100
Total steps    : 100000


## 3  Inspect one episode

Each episode is an `EpisodeData` object with numpy arrays for `observations`, `actions`,
`rewards`, `terminations`, and `truncations`.

> **Note**: Minari stores one *extra* observation per episode (the terminal observation is
> appended), so `observations` has shape `(T+1, obs_dim)` while `actions` has shape
> `(T, act_dim)`.  We use only the first `T` observations to form aligned pairs.

In [3]:
sample_episode = next(iter(dataset.iterate_episodes()))

print("observations shape :", sample_episode.observations.shape)
print("actions shape      :", sample_episode.actions.shape)
print("rewards shape      :", sample_episode.rewards.shape)

observations shape : (1001, 4)
actions shape      : (1000, 1)
rewards shape      : (1000,)


## 4  Build the 100 000 × 5 observation-action matrix

In [4]:
TARGET_ROWS = 100_000

obs_list = []
act_list = []

for episode in dataset.iterate_episodes():
    obs = episode.observations  # shape (T+1, 4)
    act = episode.actions       # shape (T, 1)
    T = act.shape[0]
    obs_list.append(obs[:T])    # align: drop terminal observation
    act_list.append(act)

    collected = sum(a.shape[0] for a in act_list)
    if collected >= TARGET_ROWS:
        break

all_obs = np.concatenate(obs_list, axis=0)   # (N, 4)
all_act = np.concatenate(act_list, axis=0)   # (N, 1)

# Trim or pad to exactly TARGET_ROWS
all_obs = all_obs[:TARGET_ROWS]
all_act = all_act[:TARGET_ROWS]

# Combine into a single (100_000, 5) matrix
hidden_state = np.hstack([all_obs, all_act])  # (100_000, 5)

print("hidden_state shape:", hidden_state.shape)
assert hidden_state.shape == (TARGET_ROWS, 5), (
    f"Expected ({TARGET_ROWS}, 5) but got {hidden_state.shape}"
)

hidden_state shape: (100000, 5)


In [18]:
# First 4 columns of hidden_state (observations only, no action)
observations = hidden_state[:, :4]  # (100_000, 4)

# 1.0 where pole angle (column 2) is within [-0.0025, 0.0025], else 0.0
pole_angle = observations[:, 2]
goal_state = np.where((pole_angle >= -0.0025) & (pole_angle <= 0.0025), 1.0, 0.0)

print("observations shape :", observations.shape)
print("goal_state shape   :", goal_state.shape)
print("goal_state=1.0 count:", int(goal_state.sum()), f"({100*goal_state.mean():.2f}%)")
idx_goal = np.where(goal_state == 1.0)[0]
print("lowest index with goal_state=1.0:", int(idx_goal[0]) if idx_goal.size > 0 else None)

observations shape : (100000, 4)
goal_state shape   : (100000,)
goal_state=1.0 count: 936 (0.94%)
lowest index with goal_state=1.0: 126


## 5  Summary statistics

In [6]:
COL_NAMES = ["cart_pos", "cart_vel", "pole_angle", "pole_ang_vel", "action"]

print(f"{'Column':<18} {'Min':>10} {'Max':>10} {'Mean':>10} {'Std':>10}")
print("-" * 62)
for i, name in enumerate(COL_NAMES):
    col = hidden_state[:, i]
    print(f"{name:<18} {col.min():>10.4f} {col.max():>10.4f} "
          f"{col.mean():>10.4f} {col.std():>10.4f}")

Column                    Min        Max       Mean        Std
--------------------------------------------------------------
cart_pos              -0.1436     0.0615    -0.0858     0.0188
cart_vel              -0.1133     0.0909    -0.0017     0.0175
pole_angle            -1.2344     0.8540    -0.0022     0.2325
pole_ang_vel          -1.8871     2.5151    -0.0000     0.5261
action                -2.8613     2.9583    -0.0000     1.0576


In [12]:
W_critsom = np.random.uniform(-1.0, 1.0, size=(10, 4))
ob = observations[0, :4]
V = np.dot(W_critsom, ob)
print("observation shape:", ob.shape)
print("observation:", ob)
print("W_critsom shape:", W_critsom.shape)
print("W_critsom:", W_critsom)
print("V shape:", V.shape)
print("V:", V)

observation shape: (4,)
observation: [ 0.00364704 -0.00892358 -0.0055928  -0.00631256]
W_critsom shape: (10, 4)
W_critsom: [[-0.42534389 -0.01808022 -0.92833576 -0.91843849]
 [-0.95506911 -0.75553919 -0.5660599  -0.36437602]
 [-0.50198787  0.93521543 -0.92532161  0.36988404]
 [ 0.25408567 -0.76947959  0.40819782  0.05901977]
 [ 0.86106127 -0.64206293  0.96053792 -0.75532598]
 [ 0.57051737 -0.43052786  0.85970757 -0.9915464 ]
 [-0.41536713 -0.90121333  0.75336126 -0.31337598]
 [ 0.2047963   0.04979996 -0.80739509 -0.48825319]
 [-0.95415728  0.24579863  0.93094967  0.19671969]
 [ 0.05725557 -0.41637958 -0.05417281 -0.79340268]]
V shape: (10,)
V: [ 0.0095998   0.00872495 -0.00733601  0.00513764  0.00826577  0.00737357
  0.00429199  0.00790024 -0.01212167  0.00923579]


In [13]:
def update_critsom(observation, W_critsom, goal_state, lr):
    """
    Update SOM prototype weights using a 1D Gaussian neighborhood.

    Parameters
    ----------
    observation : array-like, shape (4,)
        Current observation vector.
    W_critsom : np.ndarray, shape (n_neurons, 4)
        SOM prototype matrix.
    goal_state : float
        Goal indicator. If == 1.0, neuron 0 is forced to win.
    lr : float
        Learning rate.

    Returns
    -------
    W_critsom : np.ndarray
        Updated prototype matrix.
    winner : int
        Index of the winning neuron.
    output : np.ndarray
        Output vector after the update.
    """
    observation = np.asarray(observation).reshape(-1)

    if W_critsom.ndim != 2:
        raise ValueError("W_critsom must be a 2D array.")
    if observation.shape[0] != W_critsom.shape[1]:
        raise ValueError(
            f"observation must have shape ({W_critsom.shape[1]},), "
            f"got {observation.shape}"
        )

    n_neurons = W_critsom.shape[0]
    sigma = max(1.0, (n_neurons - 1) / 3.0)

    def neighborhood(center):
        idx = np.arange(n_neurons)
        return np.exp(-((idx - center) ** 2) / (2.0 * sigma**2))

    if goal_state == 1.0:
        winner = 0

        # Update forced winner
        W_critsom[0] += lr * (observation - W_critsom[0])

        # Update intermediate neurons by proximity to neuron 0
        h = neighborhood(winner)
        for i in range(1, n_neurons - 1):
            W_critsom[i] += lr * h[i] * (observation - W_critsom[i])

        # Force last neuron to be opposite of first neuron
        W_critsom[-1] = -W_critsom[0]

    else:
        # Compute output vector and identify BMU
        output = W_critsom @ observation
        winner = int(np.argmax(output))

        # Update all neurons by neighborhood proximity to BMU
        h = neighborhood(winner)[:, None]
        W_critsom += lr * h * (observation - W_critsom)

    output = W_critsom @ observation
    return W_critsom, winner, output

In [20]:
# First 4 columns of hidden_state (observations only, no action)
observations = hidden_state[:, :4]  # (100_000, 4)

# 1.0 where pole angle (column 2) is within [-0.0025, 0.0025], else 0.0
pole_angle = observations[:, 2]
goal_state = np.where((pole_angle >= -0.0025) & (pole_angle <= 0.0025), 1.0, 0.0)

W_critsom = np.random.uniform(-1.0, 1.0, size=(10, 4))
lr = 0.05

num_iterations = 5000
for step, (observation, goal) in enumerate(zip(observations[:num_iterations], goal_state[:num_iterations]), start=1):
    W_critsom, winner, output = update_critsom(observation, W_critsom, goal, lr)

    pairwise_distances = np.linalg.norm(
        W_critsom[:, None, :] - W_critsom[None, :, :],
        axis=2
    )
    neighbor_distances = np.linalg.norm(np.diff(W_critsom, axis=0), axis=1)
    if goal == 1.0:
        print(f"\nIteration {step}")
        print(f"goal_state: {goal}")
        print(f"winner neuron: {winner}")
        print("output vector:", np.round(output, 4))
        print("neighbor distances:", np.round(neighbor_distances, 4))
        print("pairwise prototype distances:")
        print(np.round(pairwise_distances, 4))


Iteration 127
goal_state: 1.0
winner neuron: 0
output vector: [ 0.0045  0.0071  0.0068  0.0068  0.0069  0.0054  0.0049  0.0036  0.0057
 -0.0045]
neighbor distances: [0.0541 0.0985 0.1178 0.1679 0.1437 0.1761 0.0884 0.0869 0.1655]
pairwise prototype distances:
[[0.     0.0541 0.0903 0.1884 0.3532 0.4933 0.6691 0.7502 0.8135 0.7784]
 [0.0541 0.     0.0985 0.2098 0.3731 0.5158 0.6916 0.7732 0.833  0.8071]
 [0.0903 0.0985 0.     0.1178 0.2811 0.423  0.5987 0.6789 0.7396 0.7145]
 [0.1884 0.2098 0.1178 0.     0.1679 0.3091 0.4852 0.5665 0.6292 0.6041]
 [0.3532 0.3731 0.2811 0.1679 0.     0.1437 0.3191 0.402  0.4624 0.4472]
 [0.4933 0.5158 0.423  0.3091 0.1437 0.     0.1761 0.2593 0.323  0.3088]
 [0.6691 0.6916 0.5987 0.4852 0.3191 0.1761 0.     0.0884 0.1533 0.1668]
 [0.7502 0.7732 0.6789 0.5665 0.402  0.2593 0.0884 0.     0.0869 0.1176]
 [0.8135 0.833  0.7396 0.6292 0.4624 0.323  0.1533 0.0869 0.     0.1655]
 [0.7784 0.8071 0.7145 0.6041 0.4472 0.3088 0.1668 0.1176 0.1655 0.    ]]

Iterati